In [ ]:
# HAIKUTHON - team-standard zero-shot evaluation: google/siglip2-large-patch16-384
# Kaggle Notebook: GPU T4 x2 (only cuda:0 is used), Internet On. Outputs go to /kaggle/working/.
# Split = team rule: np.random.RandomState(42).permutation(len(ds)), first 500 -> val.

In [ ]:
# %% [1] install (pip skips anything already satisfied, so nothing is reinstalled)
# SigLIP2 FixRes checkpoints load as model_type "siglip" -> needs transformers >= 4.49. `datasets` is preinstalled on Kaggle.
!pip install -q "transformers>=4.49.0"

In [ ]:
# %% [2] settings
MODEL_ID = "google/siglip2-large-patch16-384"
TAG = "siglip2L"                 # file-name prefix for saved outputs
OUT = "/kaggle/working"
DEVICE = "cuda:0"
MAX_LEN = 64                     # SigLIP2 text length: official usage is padding="max_length", max_length=64
BS_IMG, BS_TXT = 32, 64          # lower if CUDA OOM
LOWERCASE = True                 # HF docs: "Model was trained with lowercased text"

import os, json, time, re
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
os.makedirs(OUT, exist_ok=True)
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# %% [3] load dataset and print its structure
from datasets import load_dataset
ds = load_dataset("KimWooJin/26-haikuthon-train", split="train")
print(ds)
print("columns:", ds.column_names)
print("features:", ds.features)
CAP = next(c for c in ["prompts", "prompt", "text", "caption"] if c in ds.column_names)   # caption column
IDC = "id" if "id" in ds.column_names else None
for i in range(3):
    r = ds[i]
    print(f"\n[{i}] id={r[IDC] if IDC else i}  image size={r['image'].size}  mode={r['image'].mode}\n    {r[CAP]}")

In [ ]:
# %% [4] validation split (team rule - identical for everyone)
idx = np.random.RandomState(42).permutation(len(ds))
val_idx = sorted(idx[:500].tolist())
train_idx = sorted(idx[500:].tolist())
json.dump(val_idx, open(f"{OUT}/val_idx.json", "w"))
print("val:", len(val_idx), "train:", len(train_idx))
print("first 10 val_idx:", val_idx[:10])
if IDC: print("first 10 val ids:", [ds[i][IDC] for i in val_idx[:10]])

In [ ]:
# %% [5] model (fp16, eval) + weight size
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID, attn_implementation="sdpa").to(DEVICE).half().eval()   # T4 has no bf16 -> fp16
n_params = sum(p.numel() for p in model.parameters())
weights_gb = n_params * 2 / 1e9
print(f"params: {n_params / 1e6:.0f}M  |  fp16 weights: {weights_gb:.2f} GB")

def prep(t): return t.strip().lower() if LOWERCASE else t.strip()

In [ ]:
# %% [6] text token-length distribution (all 4000 captions; no image decoding)
caps_all = [prep(t) for t in ds[CAP]]
lens = np.array([len(ids) for ids in processor.tokenizer(caps_all)["input_ids"]])
print(f"tokens  max {lens.max()}  mean {lens.mean():.1f}  p95 {np.percentile(lens, 95):.0f}  "
      f"| truncated at {MAX_LEN}: {(lens > MAX_LEN).mean():.1%} of all, "
      f"{(lens[val_idx] > MAX_LEN).mean():.1%} of val")

In [ ]:
# %% [7] image embeddings for the 500 val images (L2-normalised, saved; reused if the file exists)
img_path = f"{OUT}/{TAG}_val_img_emb.npy"
if os.path.exists(img_path):
    img_emb = torch.from_numpy(np.load(img_path)); print("loaded", img_path, tuple(img_emb.shape))
else:
    with torch.no_grad():   # warm-up (CUDA init / kernel selection) so it does not count in the timing
        model.vision_model(pixel_values=processor(images=[ds[val_idx[0]]["image"].convert("RGB")], return_tensors="pt")["pixel_values"].to(DEVICE).half())
    cpu_t = gpu_t = 0.0; embs = []
    with torch.no_grad():
        for s in range(0, len(val_idx), BS_IMG):
            b = val_idx[s:s + BS_IMG]
            t0 = time.time()
            pv = processor(images=[ds[i]["image"].convert("RGB") for i in b], return_tensors="pt")["pixel_values"]  # decode + resize + normalize (CPU)
            cpu_t += time.time() - t0
            torch.cuda.synchronize(DEVICE); t0 = time.time()
            e = model.vision_model(pixel_values=pv.to(DEVICE).half()).pooler_output   # = get_image_features for SigLIP
            torch.cuda.synchronize(DEVICE); gpu_t += time.time() - t0
            embs.append(F.normalize(e.float(), dim=-1).cpu())
    img_emb = torch.cat(embs)
    np.save(img_path, img_emb.numpy())
    n = len(val_idx)
    gpu_ips, e2e_ips = n / gpu_t, n / (cpu_t + gpu_t)
    print(f"image encoding (this GPU): {gpu_ips:.1f} img/s GPU-only, {e2e_ips:.1f} img/s incl. CPU preprocessing")
    print(f"16,000 images on this GPU: {16000 / gpu_ips / 60:.1f} min GPU-only, {16000 / e2e_ips / 60:.1f} min incl. preprocessing (single-process)")
    # ponytail: L4 ~1.8x faster than T4 for fp16 matmul is a rough estimate, not measured; preprocessing can be parallelised with a DataLoader
    print(f"  -> rough L4 estimate (GPU part / 1.8): {16000 / (gpu_ips * 1.8) / 60:.1f} min GPU-only")
    json.dump({"gpu_img_per_s": gpu_ips, "e2e_img_per_s": e2e_ips}, open(f"{OUT}/{TAG}_speed.json", "w"))

In [ ]:
# %% [8] text embeddings for the 500 val captions (official: padding="max_length", max_length=64, truncation)
txt_path = f"{OUT}/{TAG}_val_txt_emb.npy"
val_caps = [caps_all[i] for i in val_idx]
if os.path.exists(txt_path):
    txt_emb = torch.from_numpy(np.load(txt_path)); print("loaded", txt_path, tuple(txt_emb.shape))
else:
    embs = []
    with torch.no_grad():
        for s in range(0, len(val_caps), BS_TXT):
            ids = processor(text=val_caps[s:s + BS_TXT], padding="max_length", max_length=MAX_LEN,
                            truncation=True, return_tensors="pt")["input_ids"].to(DEVICE)
            embs.append(F.normalize(model.text_model(input_ids=ids).pooler_output.float(), dim=-1).cpu())   # = get_text_features
    txt_emb = torch.cat(embs)
    np.save(txt_path, txt_emb.numpy())
print("img", tuple(img_emb.shape), "txt", tuple(txt_emb.shape))

In [ ]:
# %% [9] zero-shot retrieval: 500 texts -> 500 images
def evaluate(T, I):
    sim = T @ I.T                                      # query k's correct image is gallery column k
    d = sim.diag()[:, None]
    r = ((sim > d).sum(1) + 1).numpy().astype(float)   # ties counted in the query's favour
    ties = int(((sim == d).sum(1) > 1).sum())
    r1 = (r == 1).astype(float); nd = np.where(r <= 5, 1 / np.log2(r + 1), 0)
    return {"Recall@1": r1.mean(), "NDCG@5": nd.mean(), "FINAL": ((r1 + nd) / 2).mean(),
            "Recall@5": (r <= 5).mean(), "Recall@10": (r <= 10).mean(), "MRR": (1 / r).mean(),
            "median_rank": float(np.median(r)), "ties": ties}

res = evaluate(txt_emb, img_emb)
print("=" * 60)
print(f"model: {MODEL_ID}  | val queries: {len(val_idx)}, gallery: {len(val_idx)}")
print(f"Recall@1     : {res['Recall@1']:.4f}")
print(f"NDCG@5       : {res['NDCG@5']:.4f}")
print(f"FINAL SCORE  : {res['FINAL']:.4f}   (= mean of 0.5*(Recall@1 + NDCG@5))")
print(f"(참고) Recall@5 {res['Recall@5']:.4f} | Recall@10 {res['Recall@10']:.4f} | MRR {res['MRR']:.4f} | "
      f"median rank {res['median_rank']:.0f} | 동점 쿼리 {res['ties']}")
print(f"fp16 weights {weights_gb:.2f} GB | truncated at {MAX_LEN} tokens: {(lens[val_idx] > MAX_LEN).mean():.1%} of val")
sp = json.load(open(f"{OUT}/{TAG}_speed.json"))          # saved in cell [7], also when embeddings were reloaded
t_gpu, t_e2e = 16000 / sp["gpu_img_per_s"] / 60, 16000 / sp["e2e_img_per_s"] / 60
trunc_val = float((lens[val_idx] > MAX_LEN).mean())
json.dump({"model": MODEL_ID, "first10_val_idx": val_idx[:10], **res, "weights_gb_fp16": weights_gb,
           "trunc_ratio_val": trunc_val, "est_min_16k_gpu_T4": t_gpu, "est_min_16k_e2e_T4": t_e2e},
          open(f"{OUT}/{TAG}_zeroshot_result.json", "w"), indent=1, ensure_ascii=False)
# one line to paste into the team chat: model / first 10 idx / R@1 / NDCG@5 / score / 16k time / weights GB / truncation
print("\n[팀 공유용]")
print(f"{MODEL_ID} / {val_idx[:10]} / R@1 {res['Recall@1']:.4f} / NDCG@5 {res['NDCG@5']:.4f} / "
      f"최종 {res['FINAL']:.4f} / 16,000장 {t_gpu:.1f}분 (T4 GPU만, 전처리 포함 {t_e2e:.1f}분) / "
      f"{weights_gb:.2f} GB (fp16) / 잘림 {trunc_val:.1%} (>{MAX_LEN} tok)")

In [ ]:
# %% [10] (참고, 팀 규칙 밖) 64토큰 초과 캡션을 문장 단위로 나눠 임베딩 평균 - 잘림 대책의 효과 확인용
ntok = lambda s: len(processor.tokenizer(s)["input_ids"])
def chunked_emb(cap):
    chunks = []
    for s in re.split(r"(?<=\.)\s+", cap):
        if chunks and ntok(chunks[-1] + " " + s) <= MAX_LEN: chunks[-1] += " " + s
        else: chunks.append(s)
    ids = processor(text=chunks, padding="max_length", max_length=MAX_LEN, truncation=True, return_tensors="pt")["input_ids"].to(DEVICE)
    return F.normalize(F.normalize(model.text_model(input_ids=ids).pooler_output.float(), dim=-1).mean(0), dim=-1).cpu()
with torch.no_grad():
    txt_chunk = torch.stack([chunked_emb(c) for c in val_caps])
rc = evaluate(txt_chunk, img_emb)
print(f"(참고) sentence-chunked text: Recall@1 {rc['Recall@1']:.4f}  NDCG@5 {rc['NDCG@5']:.4f}  FINAL {rc['FINAL']:.4f}")